In [1]:
import numpy as np 
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt 

In [16]:
# Define the node class 
class Node: 
    def __init__(self , feature = None , thresold = None , left = None , right = None , * , value = None):
        """
        A tree node.
        
        Parameters:
        - feature: index of the feature to split on
        - threshold: threshold value for splitting (for numerical features)
        - left: left child node
        - right: right child node
        - value: class label if the node is a leaf
        """
        self.feature = feature # Index of the feature used for the split
        self.threshold = thresold # Threshold value for the split
        self.left = left # Left child node (<= threshold)
        self.right = right # Right child node (> threshold)
        self.value = value # If it's a leaf node, this stores the predicted class label
    
    def _is_leaf(self):
        # only leaf node has value, for others value is none 
        return self.value is not None 

In [17]:
from collections import Counter
class MyDecisionTreeClassifier: 
    def __init__(self , min_sample_split = 2 , max_depth = 100 , n_features = None): 
        """
        Decision Tree classifier.

        Parameters:
        - min_samples_split: minimum samples required to split a node
        - max_depth: maximum depth of the tree
        - n_features: number of features to consider for splits (unused here, since we use all)
        """ 
        # minimum number of samples to allow a split. If samples are less than that at current node, no split will do 
        self.min_sample_split = min_sample_split 
        self.max_depth = max_depth # max depth of the tree 
        self.n_features = n_features # Number of features to consider when splitting (for random forests)
        self.root = None 

    def fit(self , X , y): 
        """
        Build the decision tree based on input data X and target y.

        Parameters:
        - X: numpy array of shape (n_samples, n_features)
        - y: numpy array of shape (n_samples,)
        """ 
        # Determine how many features to use if not set explicitly 
        if self.n_features is None: # if n_features not given then use all columns 
            self.n_features = X.shape[1]
        else: # use the minimum if all_col and given n_features
            self.n_features = min(self.n_features , X.shape[1])

        # Build the tree recursively 
        self.root = self._grow_tree(X , y)
    def predict(self , X): 
        """
        Predict class labels for samples in X.

        Parameters:
        - X: array of samples

        Returns:
        - array of predictions
        """
        return np.array([self._traverse_tree(x , self.root) for x in X])
    
    def _traverse_tree(self , x , node): 
        """
        Traverse the tree recursively to make a prediction for sample x.

        Parameters:
        - x: single data sample
        - node: current node in the tree

        Returns:
        - predicted class label
        """ 
        if node._is_leaf(): 
            return node.value

        # Decide to go left or right based on feature threshold
        if x[node.feature] <= node.threshold: 
            return self._traverse_tree(x , node.left)
        else:
            return self._traverse_tree(x , node.right)
    def _grow_tree(self , X , y , curr_depth = 0): 
        """
        Recursively build the decision tree.

        Parameters:
        - X: data at current node
        - y: target at current node
        - depth: current depth of the tree

        Returns:
        - Node: root node of the (sub)tree
        """
        num_samples , num_features = X.shape
        num_labels = len(np.unique(y))
        # Stopping conditions 
        ''' 
        if curr_depth >= max_depth --> dont go deeper, STOP
        if pure split(means we only have one class at this level) then stop it's a leaf node
        if curr_total samples < min_sample then we are not allowed to go deep
        '''
        if(curr_depth >= self.max_depth or num_labels == 1 or num_samples < self.min_sample_split): 
            # make a leaf node 
            # predict the label 
            leaf_value = self._most_common_label(y)
            leaf_node = Node(value = leaf_value)
            return leaf_node

        # choose n_features random feature where we will do the split 
        feature_indices = np.random.choice(num_features , self.n_features , replace = False)
        
        # Find the best feature and threshold to split on
        best_feature , best_threshold = self._best_split(X , y , feature_indices)

        # If no valid split found, make leaf node 
        if best_feature is None:
            leaf_value = self._most_common_label(y)
            leaf_node = Node(value = leaf_value)
            return leaf_node

        # Split data based on best split using _split
        X_column = X[ : , best_feature]
        left_indices , right_indices = self._split(X_column , best_threshold)

        # Recursively build left and right subtrees
        left = self._grow_tree(X[left_indices , : ] , y[left_indices] , curr_depth + 1) 
        right = self._grow_tree(X[right_indices , : ] , y[right_indices] , curr_depth + 1) 

        # Return the current node with children 
        curr_node = Node(best_feature , best_threshold , left , right)
        return curr_node

    def _best_split(self , X , y , feature_indices): 
        """
        Find the best feature and threshold to split the data.

        Parameters:
        - X: data at current node
        - y: target at current node
        - feat_idxs: indices of features to consider

        Returns:
        - split_idx: feature index for best split
        - split_threshold: threshold value for best split
        """
        best_gain = -1 
        split_index , split_threshold = None , None 

        # go to every feature and split 
        for feature_index in feature_indices: 
            # get the feature col values
            X_column = X[ : , feature_index]
            # find the unique values baed on what we can split 
            thresholds = np.unique(X_column)

            # Evaluate each threshold
            for threshold in thresholds: 
                # find the gain for current threshold 
                curr_gain = self._information_gain(y , X_column , threshold)

                # if curr_gain is higher than best_gain then split based on current gain using current threshold 
                if curr_gain > best_gain: 
                    best_gain = curr_gain
                    split_index = feature_index
                    split_threshold = threshold
                    
        return split_index , split_threshold 
    def _information_gain(self , y , X_column , threshold): 
        """
        Calculate Information Gain of a split.
        
        IG = Entropy(parent) - [weighted average] * Entropy(childrens)
        
        Parameters:
        - y: target at current node
        - X_column: values of feature column
        - threshold: value to split on

        Returns:
        - information_gain: how much entropy is reduced by the split
        """
        # find parent entropy
        parent_entropy = self._entropy(y) 
        # find(create) childrens
        left_indices , right_indices = self._split(X_column , threshold)
        # if left_indices or right_indices has empty then IG is 0 
        # No split if one side is empty
        if len(left_indices) == 0 or len(right_indices) == 0: 
            return 0
            
        # childrens weighted entropy
        n = len(y) # Total number of samples before the split (in the current node) ==> count of sample in parent(S).
        # Number of samples in the left child node and right child node after the split.
        n_l , n_r = len(left_indices) , len(right_indices) 
        # Find the entropy of left and right child 
        left_entropy , right_entropy = self._entropy(y[left_indices]) , self._entropy(y[right_indices])
        # Compute the weighted average entropy of the children:
        child_entropy = ((n_l / n) * left_entropy) + ((n_r / n) * right_entropy)
        # calculate IG 
        # Information gain is parent entropy minus weighted children entropy
        information_gain = parent_entropy - child_entropy
        return information_gain

    def _split(self , X_column , threshold):  
        """
        Split data indices based on threshold.

        Parameters:
        - X_column: feature column
        - split_thresh: threshold to split on

        Returns:
        - left_idxs: indices where feature <= threshold
        - right_idxs: indices where feature > threshold
        """
        left_indices = np.argwhere(X_column <= threshold).flatten()
        right_indices = np.argwhere(X_column > threshold).flatten()
        return  left_indices , right_indices
    def _entropy(self , y): 
        """
        Calculate entropy of label array y.

        Parameters:
        - y: target values

        Returns:
        - entropy value
        """
        # count the frequency of each class 
        hist = np.bincount(y)
        # find the probability of each class: p(x) = x_freq / total_freq 
        probabilities = hist / len(y) 
        entropy = np.sum([-p * np.log2(p) for p in probabilities if p > 0])
        return entropy
    def _most_common_label(self , y): 
        """
        Find the most common label in y.

        Parameters:
        - y: target values

        Returns:
        - most frequent label
        """ 
        count_labels = Counter(y)
        most_common_label = count_labels.most_common(1)[0][0]
        return most_common_label

In [18]:
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [19]:
# Load the breast cancer data 
data = datasets.load_breast_cancer()
X , y = data.data , data.target

In [20]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size = 0.2 , random_state = 42)

In [21]:
our_model = MyDecisionTreeClassifier()
our_model.fit(X_train , y_train)

y_pred = our_model.predict(X_test)
y_pred

array([1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1,
       0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1,
       1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1,
       0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0,
       1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1,
       0, 1, 1, 0])

In [22]:
print(f"Our model Accuracy: {accuracy_score(y_test , y_pred)}")

Our model Accuracy: 0.9298245614035088


In [23]:
from sklearn.tree import DecisionTreeClassifier

sk_model = DecisionTreeClassifier()
sk_model.fit(X_train , y_train)

y_pred = sk_model.predict(X_test)
y_pred

array([1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1,
       0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1,
       1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1,
       0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0,
       1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1,
       0, 1, 1, 0])

In [24]:
print(f"sklearb model Accuracy: {accuracy_score(y_test , y_pred)}")

sklearb model Accuracy: 0.9298245614035088
